In [0]:
# ---------------------------------------------
# Silver Layer ETL for IoT Sensor Data
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, IntegerType, DoubleType

try:

    # Read Bronze table
    df = spark.table("iot_sensor_anomoly_detection.bronze.device_health_diagnostics")

    silver_df = df.select(

        # Device ID (primary identifier)
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # Handle NULL + empty strings for health_indicator
        upper(
            coalesce(
                when(trim(col("health_indicator")) == "", None)
                .otherwise(trim(col("health_indicator"))),
                lit("UNKNOWN")
            )
        ).alias("health_indicator"),

        # Firmware status cleaning
        upper(
            coalesce(
                when(trim(col("firmware_status")) == "", None)
                .otherwise(trim(col("firmware_status"))),
                lit("UNKNOWN")
            )
        ).alias("firmware_status"),

        # Maintenance group cleaning
        upper(
            coalesce(
                when(trim(col("maintenance_group")) == "", None)
                .otherwise(trim(col("maintenance_group"))),
                lit("UNKNOWN")
            )
        ).alias("maintenance_group"),

        # Handle NULL + empty strings for support_team
        upper(
            coalesce(
                when(trim(col("support_team")) == "", None)
                .otherwise(trim(col("support_team"))),
                lit("UNKNOWN")
            )
        ).alias("support_team"),

        # Diagnostic flag cleaning
        upper(
            coalesce(
                when(trim(col("diagnostic_flag")) == "", None)
                .otherwise(trim(col("diagnostic_flag"))),
                lit("UNKNOWN")
            )
        ).alias("diagnostic_flag"),

        # Handle numeric NULL values
        coalesce(col("health_score").cast(DoubleType()), lit(0.0)).alias("health_score"),

        coalesce(col("battery_voltage").cast(DoubleType()), lit(0.0)).alias("battery_voltage"),

        coalesce(col("error_count").cast(IntegerType()), lit(0)).alias("error_count"),

        coalesce(col("restart_count").cast(IntegerType()), lit(0)).alias("restart_count"),

        coalesce(col("maintenance_cost").cast(DoubleType()), lit(0.0)).alias("maintenance_cost")
    )

    # Remove invalid records
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # Remove duplicates
    silver_df = silver_df.dropDuplicates(["device_id"])

    # Write Silver table
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_sensor_anomoly_detection.silver.device_health_diagnostics_clean")

    print("✅ Silver table created successfully")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.silver.device_health_diagnostics_clean
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for device_operations
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, DoubleType

try:

    # Read Bronze table
    df = spark.table("iot_sensor_anomoly_detection.bronze.device_operations")

    silver_df = df.select(

        # Device ID (primary identifier)
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # Device category cleaning
        upper(
            coalesce(
                when(trim(col("device_category")) == "", None)
                .otherwise(trim(col("device_category"))),
                lit("UNKNOWN")
            )
        ).alias("device_category"),

        # Maker name cleaning
        upper(
            coalesce(
                when(trim(col("maker_name")) == "", None)
                .otherwise(trim(col("maker_name"))),
                lit("UNKNOWN")
            )
        ).alias("maker_name"),

        # Firmware build cleaning
        upper(
            coalesce(
                when(trim(col("firmware_build")) == "", None)
                .otherwise(trim(col("firmware_build"))),
                lit("UNKNOWN")
            )
        ).alias("firmware_build"),

        # Production line cleaning
        upper(
            coalesce(
                when(trim(col("production_line")) == "", None)
                .otherwise(trim(col("production_line"))),
                lit("UNKNOWN")
            )
        ).alias("production_line"),

        # Factory site cleaning
        upper(
            coalesce(
                when(trim(col("factory_site")) == "", None)
                .otherwise(trim(col("factory_site"))),
                lit("UNKNOWN")
            )
        ).alias("factory_site"),

        # Numeric column cleaning
        coalesce(col("battery_pct").cast(DoubleType()), lit(0.0)).alias("battery_pct"),

        coalesce(col("cpu_load").cast(DoubleType()), lit(0.0)).alias("cpu_load"),

        coalesce(col("memory_usage").cast(DoubleType()), lit(0.0)).alias("memory_usage"),

        coalesce(col("device_temp").cast(DoubleType()), lit(0.0)).alias("device_temp"),

        coalesce(col("uptime_hours").cast(DoubleType()), lit(0.0)).alias("uptime_hours")
    )

    # Remove invalid records
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # Remove duplicates
    silver_df = silver_df.dropDuplicates(["device_id"])

    # Write Silver table
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_sensor_anomoly_detection.silver.device_operations_clean")

    print("✅ device_operations Silver table created successfully")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.silver.device_operations_clean
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for environment_network
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, DoubleType

try:

    # Read Bronze table
    df = spark.table("iot_sensor_anomoly_detection.bronze.environment_network")

    silver_df = df.select(

        # Device ID
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # Region name cleaning
        upper(
            coalesce(
                when(trim(col("region_name")) == "", None)
                .otherwise(trim(col("region_name"))),
                lit("UNKNOWN")
            )
        ).alias("region_name"),

        # City name cleaning
        upper(
            coalesce(
                when(trim(col("city_name")) == "", None)
                .otherwise(trim(col("city_name"))),
                lit("UNKNOWN")
            )
        ).alias("city_name"),

        # Site location cleaning
        upper(
            coalesce(
                when(trim(col("site_location")) == "", None)
                .otherwise(trim(col("site_location"))),
                lit("UNKNOWN")
            )
        ).alias("site_location"),

        # Network provider cleaning
        upper(
            coalesce(
                when(trim(col("network_provider")) == "", None)
                .otherwise(trim(col("network_provider"))),
                lit("UNKNOWN")
            )
        ).alias("network_provider"),

        # Weather condition cleaning
        upper(
            coalesce(
                when(trim(col("weather_condition")) == "", None)
                .otherwise(trim(col("weather_condition"))),
                lit("UNKNOWN")
            )
        ).alias("weather_condition"),

        # Numeric columns null handling
        coalesce(col("ambient_temp").cast(DoubleType()), lit(0.0)).alias("ambient_temp"),

        coalesce(col("humidity_pct").cast(DoubleType()), lit(0.0)).alias("humidity_pct"),

        coalesce(col("network_latency").cast(DoubleType()), lit(0.0)).alias("network_latency"),

        coalesce(col("packet_loss_rate").cast(DoubleType()), lit(0.0)).alias("packet_loss_rate"),

        coalesce(col("signal_db").cast(DoubleType()), lit(0.0)).alias("signal_db")
    )

    # Remove invalid rows
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # Remove duplicates
    silver_df = silver_df.dropDuplicates(["device_id"])

    # Write Silver table
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_sensor_anomoly_detection.silver.environment_network_clean")

    print("✅ Silver table created successfully")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.silver.environment_network_clean
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for sensor_stream
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when
from pyspark.sql.types import StringType, DoubleType

try:

    # Read Bronze table
    df = spark.table("iot_sensor_anomoly_detection.bronze.sensor_stream")

    silver_df = df.select(

        # Device ID
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # Sensor identifier
        trim(col("sensor_identifier")).cast(StringType()).alias("sensor_identifier"),

        # Sensor category cleaning
        upper(
            coalesce(
                when(trim(col("sensor_category")) == "", None)
                .otherwise(trim(col("sensor_category"))),
                lit("UNKNOWN")
            )
        ).alias("sensor_category"),

        # Reading unit cleaning
        upper(
            coalesce(
                when(trim(col("reading_unit")) == "", None)
                .otherwise(trim(col("reading_unit"))),
                lit("UNKNOWN")
            )
        ).alias("reading_unit"),

        # Sensor model cleaning
        upper(
            coalesce(
                when(trim(col("sensor_model")) == "", None)
                .otherwise(trim(col("sensor_model"))),
                lit("UNKNOWN")
            )
        ).alias("sensor_model"),

        # Sensor zone cleaning
        upper(
            coalesce(
                when(trim(col("sensor_zone")) == "", None)
                .otherwise(trim(col("sensor_zone"))),
                lit("UNKNOWN")
            )
        ).alias("sensor_zone"),

        # Numeric columns
        coalesce(col("reading_value").cast(DoubleType()), lit(0.0)).alias("reading_value"),

        coalesce(col("threshold_low").cast(DoubleType()), lit(0.0)).alias("threshold_low"),

        coalesce(col("threshold_high").cast(DoubleType()), lit(0.0)).alias("threshold_high"),

        coalesce(col("signal_strength").cast(DoubleType()), lit(0.0)).alias("signal_strength"),

        coalesce(col("noise_level").cast(DoubleType()), lit(0.0)).alias("noise_level")
    )

    # Remove invalid rows
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # Remove duplicates
    silver_df = silver_df.dropDuplicates(["sensor_identifier"])

    # Write Silver table
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_sensor_anomoly_detection.silver.sensor_stream_clean")

    print("✅ sensor_stream Silver table created successfully")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.silver.sensor_stream_clean
LIMIT 100;

In [0]:
# ---------------------------------------------
# Silver Layer ETL for time_anomaly_events
# ---------------------------------------------
from pyspark.sql.functions import col, trim, upper, coalesce, lit, when, expr
from pyspark.sql.types import StringType, IntegerType, DoubleType

# Helper function to safely convert timestamp
def safe_timestamp(column_name):
    return expr(f"try_to_timestamp({column_name})")

try:

    # Read Bronze table
    df = spark.table("iot_sensor_anomoly_detection.bronze.time_anomaly_events")

    silver_df = df.select(

        # Device ID
        trim(col("device_id")).cast(StringType()).alias("device_id"),

        # Convert timestamp
        safe_timestamp("event_timestamp").alias("event_timestamp"),

        # Event type cleaning
        upper(
            coalesce(
                when(trim(col("event_type")) == "", None)
                .otherwise(trim(col("event_type"))),
                lit("UNKNOWN")
            )
        ).alias("event_type"),

        # Anomaly category cleaning
        upper(
            coalesce(
                when(trim(col("anomaly_category")) == "", None)
                .otherwise(trim(col("anomaly_category"))),
                lit("UNKNOWN")
            )
        ).alias("anomaly_category"),

        # Severity level cleaning
        upper(
            coalesce(
                when(trim(col("severity_level")) == "", None)
                .otherwise(trim(col("severity_level"))),
                lit("UNKNOWN")
            )
        ).alias("severity_level"),

        # Root cause cleaning
        upper(
            coalesce(
                when(trim(col("root_cause_hint")) == "", None)
                .otherwise(trim(col("root_cause_hint"))),
                lit("UNKNOWN")
            )
        ).alias("root_cause_hint"),

        # Numeric columns
        coalesce(col("downtime_minutes").cast(IntegerType()), lit(0)).alias("downtime_minutes"),

        coalesce(col("event_duration").cast(DoubleType()), lit(0.0)).alias("event_duration"),

        coalesce(col("retry_count").cast(IntegerType()), lit(0)).alias("retry_count"),

        coalesce(col("error_code").cast(IntegerType()), lit(0)).alias("error_code"),

        coalesce(col("event_score").cast(DoubleType()), lit(0.0)).alias("event_score")
    )

    # Remove invalid rows
    silver_df = silver_df.filter(col("device_id").isNotNull())

    # Remove duplicates based on device_id + event_timestamp
    silver_df = silver_df.dropDuplicates(["device_id","event_timestamp"])

    # Write Silver table
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("iot_sensor_anomoly_detection.silver.time_anomaly_events_clean")

    print("✅ time_anomaly_events Silver table created successfully")

except Exception as e:
    print(f"❌ Error in Silver layer: {e}")

In [0]:
%sql
SELECT *
FROM iot_sensor_anomoly_detection.silver.time_anomaly_events_clean
LIMIT 100;